# LoRA Implementation

This notebook implements a series of exercises focused on **Parameter‑Efficient Fine‑Tuning (PEFT)** using the **LoRA** (Low‑Rank Adaptation) technique. LoRA introduces learnable low‑rank matrices $A$ and $B$ to existing linear layers, allowing models to be fine‑tuned efficiently without updating the entire weight matrix.

These exercises walk through building LoRA components from scratch, integrating them into a simple network, and preparing a LoRA‑aware MLP that freezes the original layers.

> **Note:** The code provided here illustrates how LoRA layers can be built and integrated. Because the current environment does not include the `torch` library, and training a full model is computationally intensive, the code cells are not executed. You can run this notebook locally with PyTorch installed to verify and experiment.


## Exercise 1 – Implementing the `LoRALayer`

**Objective:**

Create a custom PyTorch module called **`LoRALayer`** that introduces low‑rank adaptation matrices $A$ and $B$.

**Instructions:**

1. Create a PyTorch class `LoRALayer` inheriting from `nn.Module`.
2. In the `__init__` method, initialize the low‑rank matrices `A` and `B` with shapes `(r, in_features)` and `(out_features, r)` where `r` is the rank of the adaptation.
3. Implement the `forward` method to compute the LoRA transformation:  \(	ext{LoRA}(x) = x + B(A(x))\).
4. Test the class with a small input tensor to verify its functionality.


In [ ]:
import torch
import torch.nn as nn

# A simple LoRA layer that adds a low‑rank adaptation to a linear transformation
class LoRALayer(nn.Module):
    def __init__(self, in_features: int, out_features: int, rank: int = 4):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        # Original weight matrix
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        # Low‑rank matrices A (rank x in_features) and B (out_features x rank)
        self.A = nn.Parameter(torch.randn(rank, in_features))
        self.B = nn.Parameter(torch.randn(out_features, rank))
        # Optional bias
        self.bias = nn.Parameter(torch.zeros(out_features))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_output = x @ self.weight.t() + self.bias
        adaptation = (x @ self.A.t()) @ self.B.t()
        return base_output + adaptation

# Example usage (disabled by default)
if False:
    layer = LoRALayer(in_features=8, out_features=4, rank=2)
    x = torch.randn(3, 8)
    y = layer(x)
    print(y.shape)


## Exercise 2 – Implementing the `LinearWithLoRA` Layer

**Objective:**

Extend a standard PyTorch `Linear` layer to incorporate LoRA for adaptable training.

**Instructions:**

1. Create a new class `LinearWithLoRA` that wraps an existing `nn.Linear` layer.
2. Add an instance of `LoRALayer` to introduce low‑rank adaptation.
3. Implement the `forward` method to return the sum of the standard linear transformation and the LoRA adaptation.
4. Test this new layer with an input tensor.


In [ ]:
class LinearWithLoRA(nn.Module):
    # A linear layer augmented with a LoRA adaptation
    def __init__(self, in_features: int, out_features: int, rank: int = 4):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features, bias=True)
        self.lora = LoRALayer(in_features, out_features, rank=rank)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(x) + self.lora(x)

# Example usage (disabled)
if False:
    layer = LinearWithLoRA(in_features=8, out_features=4, rank=2)
    x = torch.randn(3, 8)
    y = layer(x)
    print(y.shape)


## Exercise 3 – Creating a Small Neural Network and Applying LoRA

**Objective:**

Implement a simple feedforward neural network and apply LoRA to one of its layers.

**Instructions:**

1. Define a single‑layer neural network using `nn.Linear`.
2. Generate a random input tensor to test the layer.
3. Replace the `Linear` layer with `LinearWithLoRA` and verify that the outputs remain unchanged initially (before training).


In [ ]:
class SimpleNN(nn.Module):
    def __init__(self, in_features: int, hidden_features: int, out_features: int, use_lora: bool = False, rank: int = 4):
        super().__init__()
        if use_lora:
            self.fc = LinearWithLoRA(in_features, hidden_features, rank=rank)
        else:
            self.fc = nn.Linear(in_features, hidden_features)
        self.out = nn.Linear(hidden_features, out_features)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.relu(self.fc(x))
        return self.out(x)

# Example usage (disabled)
if False:
    model_plain = SimpleNN(in_features=8, hidden_features=16, out_features=4, use_lora=False)
    model_lora = SimpleNN(in_features=8, hidden_features=16, out_features=4, use_lora=True, rank=2)
    x = torch.randn(3, 8)
    out_plain = model_plain(x)
    out_lora = model_lora(x)


## Exercise 4 – Merging LoRA Matrices and Testing Equivalence

**Objective:**

Implement an alternative approach where LoRA matrices are merged with the original weights for efficiency.

**Instructions:**

1. Create a new class `LinearWithLoRAMerged` that computes the combined weight matrix `(W + BA)` on the fly.
2. Ensure that the output remains the same as `LinearWithLoRA` for a given input.
3. Test with a sample input to verify correctness.


In [ ]:
class LinearWithLoRAMerged(nn.Module):
    # Linear layer where the LoRA matrices are merged with the original weight matrix
    def __init__(self, in_features: int, out_features: int, rank: int = 4):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        self.A = nn.Parameter(torch.randn(rank, in_features))
        self.B = nn.Parameter(torch.randn(out_features, rank))
        self.bias = nn.Parameter(torch.zeros(out_features))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        merged_weight = self.weight + self.B @ self.A
        return x @ merged_weight.t() + self.bias

# Example usage (disabled)
if False:
    merged = LinearWithLoRAMerged(in_features=8, out_features=4, rank=2)
    x = torch.randn(3, 8)
    out = merged(x)


## Exercise 5 – Implementing a Multilayer Perceptron (MLP) and Replacing Layers with LoRA

**Objective:**

Extend a simple MLP and modify its layers to use LoRA.

**Instructions:**

1. Implement a 3‑layer MLP.
2. Replace each `Linear` layer with `LinearWithLoRAMerged`.
3. Print the model architecture to verify the modifications.


In [ ]:
class LoRA_MLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: list[int], output_dim: int, rank: int = 4):
        super().__init__()
        layers = []
        in_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(LinearWithLoRAMerged(in_dim, h_dim, rank=rank))
            layers.append(nn.ReLU())
            in_dim = h_dim
        layers.append(LinearWithLoRAMerged(in_dim, output_dim, rank=rank))
        self.model = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)

# Example usage (disabled)
if False:
    mlp = LoRA_MLP(input_dim=8, hidden_dims=[16, 16], output_dim=4, rank=2)
    print(mlp)


## Exercise 6 – Freezing the Original Linear Layers and Training LoRA

**Objective:**

Ensure only LoRA layers are trainable and train the model.

**Instructions:**

1. Implement a function to freeze standard `Linear` layers in a model (set `requires_grad = False`).
2. Apply it to the MLP model.
3. Print trainable parameters to confirm only LoRA layers are trainable.
4. *(Optional)* Train the model on a dataset and evaluate its performance. Training a full model requires a dataset and compute resources; the code below provides the structure for your environment.


In [ ]:
def freeze_linear_layers(model: nn.Module) -> None:
    # Freeze all standard linear layers in the model
    for module in model.modules():
        if isinstance(module, nn.Linear):
            for param in module.parameters():
                param.requires_grad = False

# Example of freezing (disabled)
if False:
    mlp = LoRA_MLP(input_dim=8, hidden_dims=[16, 16], output_dim=4, rank=2)
    freeze_linear_layers(mlp)
    # List trainable parameters
    for name, param in mlp.named_parameters():
        if param.requires_grad:
            print(name)
